# Module 3 — Fast EDA & Data Health
**Type:** [Code With Me — Visualizations]  
**Time:** 15 minutes  
**Job:** Spot seasonality, zeros, and intermittency. Validate the subset. Save the cleaned panel artifact.

---
## 3.1 — Setup
**[Watch Only]**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from config import (
    WORKSHOP_SUBSET_PATH,
    ARTIFACT_DIR,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    SEASON_LENGTH,
    MAX_ZERO_FRACTION,
    HORIZON,
    N_WINDOWS,
    STEP_SIZE,
)
from src.schemas import validate_panel

print("Setup complete.")

---
## 3.2 — Load the Panel
**[Watch Only]**

In [ ]:
panel = pd.read_parquet(WORKSHOP_SUBSET_PATH)
panel = validate_panel(panel, artifact_name="m5_workshop_subset")
panel["ds"] = pd.to_datetime(panel["ds"])

n_series = panel["unique_id"].nunique()
n_rows   = len(panel)
min_date = panel["ds"].min()
max_date = panel["ds"].max()

print(f"Panel loaded")
print(f"  Series     : {n_series:,}")
print(f"  Rows       : {n_rows:,}")
print(f"  Date range : {min_date.date()} → {max_date.date()}")
print(f"  Columns    : {list(panel.columns)}")

**Expected output:**
```
Panel loaded
  Series     : 1,000
  Rows       : ~1,913,000
  Date range : 2011-01-29 → 2016-06-19
  Columns    : ['unique_id', 'ds', 'y']
```

---
## 3.3 — Select the Micro Subset for Live Exploration
**[Watch Only]**

In [ ]:
# Select MICRO_SUBSET_N series by total sales volume — deterministic, matches subset policy
top_series = (
    panel.groupby("unique_id")["y"]
    .sum()
    .sort_values(ascending=False)
    .head(MICRO_SUBSET_N)
    .index
)

micro = panel[panel["unique_id"].isin(top_series)].copy()

print(f"Micro subset: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 3.4 — Plot Sample Series
**[Code With Me — 1 line]**

Fill in the plot call inside the loop. Plot `s.index` on the x-axis and `s.values` on the y-axis.

In [ ]:
# Plot the first 4 series from the micro subset
sample_ids = top_series[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 6))
axes = axes.flatten()

for i, uid in enumerate(sample_ids):
    s = micro[micro["unique_id"] == uid].set_index("ds")["y"]
    axes[i].plot(__FILL_IN__, linewidth=0.8, color="steelblue")  # plot s.index vs s.values
    axes[i].set_title(uid, fontsize=9)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[i].xaxis.set_major_locator(mdates.YearLocator())
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Units sold")

plt.suptitle("Sample Series — Daily Sales (Full History)", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**Expected output:** Four time series plots showing daily sales from 2011 to 2016. You should see weekly oscillation in all four and varying scale across series.

---
## 3.5 — Seasonality Check: Average Weekly Profile
**[Watch Only]**

In [ ]:
# Average sales by day of week across the full panel
panel["dow"] = panel["ds"].dt.dayofweek  # 0=Monday, 6=Sunday
dow_profile = panel.groupby("dow")["y"].mean()

day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(day_labels, dow_profile.values, color="steelblue", alpha=0.85)
ax.set_title("Average Daily Sales by Day of Week (Full Panel)", fontsize=11)
ax.set_ylabel("Mean units sold")
ax.axhline(dow_profile.mean(), color="tomato", linestyle="--", linewidth=1, label="Panel mean")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Peak day  : {day_labels[dow_profile.idxmax()]} ({dow_profile.max():.1f} units avg)")
print(f"Trough day: {day_labels[dow_profile.idxmin()]} ({dow_profile.min():.1f} units avg)")
print(f"Peak/trough ratio: {dow_profile.max() / dow_profile.min():.2f}x")

**Expected output:** A bar chart showing a clear weekly pattern with a peak/trough ratio > 1.5x.

---
## 3.6 — Intermittency Check
**[Code With Me — 1 line]**

Fill in the lambda to compute the fraction of zero-demand days per series.

In [ ]:
# Compute fraction of zero-demand days per series
zero_frac = (
    panel.groupby("unique_id")["y"]
    .apply(__FILL_IN__)  # lambda: fraction of zeros per series
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(zero_frac.values, bins=40, color="steelblue", alpha=0.85, edgecolor="white")
ax.axvline(MAX_ZERO_FRACTION, color="tomato", linestyle="--", linewidth=1.5,
           label=f"Filter threshold ({int(MAX_ZERO_FRACTION*100)}%)")
ax.set_title("Distribution of Zero-Demand Fraction Across Series", fontsize=11)
ax.set_xlabel("Fraction of zero-demand days")
ax.set_ylabel("Number of series")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

n_high_zero = (zero_frac > 0.30).sum()
print(f"Series with > 30% zero-demand days: {n_high_zero} of {len(zero_frac):,}")
print(f"Max zero fraction in subset       : {zero_frac.max():.2%}")
print(f"Median zero fraction              : {zero_frac.median():.2%}")

**Expected output:** Histogram concentrated below 30%. No series should exceed the 50% filter threshold (dashed red line).

---
## 3.7 — History Depth Check
**[Watch Only]**

In [ ]:
# Verify every series has sufficient history before the first CV cutoff
end_date     = panel["ds"].max()
first_cutoff = end_date - pd.Timedelta(days=(N_WINDOWS * STEP_SIZE))
min_start    = first_cutoff - pd.Timedelta(days=365)

series_starts = panel.groupby("unique_id")["ds"].min()
insufficient  = series_starts[series_starts > min_start]

print(f"End date         : {end_date.date()}")
print(f"First CV cutoff  : {first_cutoff.date()}")
print(f"Min start needed : {min_start.date()}")
print()

if len(insufficient) > 0:
    print(f"  ✗ WARNING — {len(insufficient)} series have insufficient history:")
    print(insufficient.head(5).to_string())
else:
    print(f"  ✓ All {n_series:,} series have ≥ 365 days of history before the first cutoff.")

**Expected output:**
```
End date         : 2016-06-19
First CV cutoff  : 2016-03-27
Min start needed : 2015-03-27

  ✓ All 1,000 series have ≥ 365 days of history before the first cutoff.
```

---
## 3.8 — CV Cutoff Visualization
**[Watch Only]**

In [ ]:
# Visualize the 3 CV windows on a single representative series
sample_uid  = top_series[0]
sample_ts   = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]

cutoffs = [
    end_date - pd.Timedelta(days=(N_WINDOWS - i) * STEP_SIZE)
    for i in range(N_WINDOWS)
]

# Show only the last 180 days of history for readability
plot_start = cutoffs[0] - pd.Timedelta(days=60)
sample_plot = sample_ts[sample_ts.index >= plot_start]

colors = ["#2196F3", "#FF9800", "#4CAF50"]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample_plot.index, sample_plot.values, color="#555", linewidth=0.9, zorder=2)

for i, (cutoff, color) in enumerate(zip(cutoffs, colors)):
    horizon_end = cutoff + pd.Timedelta(days=HORIZON)
    ax.axvline(cutoff,      color=color, linestyle="--", linewidth=1.2, alpha=0.9)
    ax.axvspan(cutoff, horizon_end, alpha=0.12, color=color,
               label=f"Window {i+1}: {cutoff.date()} + {HORIZON}d")

ax.set_title(f"CV Windows on {sample_uid}", fontsize=11)
ax.set_ylabel("Units sold")
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()

print("Cross-validation cutoffs:")
for i, c in enumerate(cutoffs):
    print(f"  Window {i+1}: cutoff={c.date()}  forecast window: {c.date()} → {(c + pd.Timedelta(days=HORIZON)).date()}")

**Expected output:** A time series chart with three shaded forecast windows, each 28 days wide, spaced 28 days apart.

---
## 3.9 — Save the Validated Panel Artifact
**[Watch Only]**

In [ ]:
from src.schemas import validate_panel

# Drop EDA helper column before saving
panel_clean = panel.drop(columns=["dow"], errors="ignore")
panel_clean = validate_panel(panel_clean, artifact_name="03_validated_panel")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
artifact_path = ARTIFACT_DIR / "03_validated_panel.parquet"
panel_clean.to_parquet(artifact_path, index=False)

print(f"  ✓ Validated panel saved: {artifact_path.name}")
print(f"    Rows   : {len(panel_clean):,}")
print(f"    Series : {panel_clean['unique_id'].nunique():,}")

**Expected output:**
```
  ✓ Validated panel saved: 03_validated_panel.parquet
    Rows   : ~1,913,000
    Series : 1,000
```

---
## 3.10 — Red Path Recovery
**[Watch Only]**

In [ ]:
# 🔴 RED PATH — run this cell only if the Green Path above failed
# Loads the precomputed validated panel from artifacts/

# from src.checkpointing import load_checkpoint
# panel_clean = load_checkpoint("03_validated_panel")

---
## 3.11 — The Enterprise Cliff
**[Watch Only]**

We validated four things in 15 minutes: schema, series count, zero fraction, and history depth.

In a production environment, data health takes weeks, not minutes. What we skipped:

- **Stockout detection.** A run of zeros may be intermittent demand or a supply gap. Evaluating against stockout periods inflates error metrics artificially — you are penalizing the model for events it could not have predicted.
- **Price change alignment.** Sell price shifts in the M5 dataset are available but unused here. In production, a price increase that tanks demand is a covariate event, not a forecast failure.
- **Calendar anomaly exclusion.** Thanksgiving, Easter, and regional holidays create spikes that skew CV window scores. Enterprise pipelines maintain exclusion calendars that are updated before every evaluation run.

Clean data health is not a data engineering problem. It is a forecasting accuracy problem. Every undetected anomaly in your training set is a structural bias you will carry into production.